# 01 — Item Vector (TF-IDF with Weighted Soup)

Notebook 2/4 of the Content-Based pipeline.

**Configurable Parameters**
| Parameter | Description |
|-----------|-------------|
| `VERSION` | Data version to process (e.g., 'v2') |

**Output Format:** `cb_tfidf_item_vector_{VERSION}.npz`, `cb_tfidf_vectorizer_{VERSION}.pkl`, `cb_soup_texts_{VERSION}.parquet`

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import scipy.sparse as sp
import pyarrow as pa
import pyarrow.parquet as pq
import pickle, gc, os, re, psutil
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm import tqdm

def print_ram(label=''):
    mem = psutil.Process(os.getpid()).memory_info().rss / 1024**3
    tag = f' [{label}]' if label else ''
    print(f'  [RAM{tag}] {mem:.2f} GB')

# ============================================================
# CONFIG
# ============================================================
VERSION = 'v5'  # Change this to switch versions

BASE_DIR   = '/content/drive/My Drive/Thesis/book_recsys'
DATA_DIR   = os.path.join(BASE_DIR, 'data/processed')

BOOKS_PQ   = os.path.join(DATA_DIR, f'cb_books_{VERSION}.parquet')
SOUP_OUT   = os.path.join(DATA_DIR, f'cb_soup_texts_{VERSION}.parquet')
TFIDF_VEC  = os.path.join(DATA_DIR, f'cb_tfidf_vectorizer_{VERSION}.pkl')
TFIDF_MAT  = os.path.join(DATA_DIR, f'cb_tfidf_item_vector_{VERSION}.npz')

SOUP_CHUNK   = 50_000
TFIDF_CHUNK  = 50_000
SAMPLE_SIZE  = 300_000
MAX_FEATURES = 20_000

print(f'Config done for VERSION: {VERSION}')
print_ram()

Config done for VERSION: v5
  [RAM] 0.23 GB


## 1. Build Weighted Soup

In [3]:
NOISE_SHELVES = {'to-read', 'currently-reading', 'owned', 'books-i-own', 'favourites', 'favorite', 'default', 'to-buy', 'maybe', 'wish-list', 'library', 'ebook', 'kindle'}

def clean_text(text):
    if pd.isna(text) or str(text).strip() in ('', 'nan'): return ''
    t = str(text).lower().replace('-', ' ')
    t = re.sub(r'[^a-z0-9\s]', ' ', t)
    return re.sub(r'\s+', ' ', t).strip()

def clean_shelves(shelves_str):
    if pd.isna(shelves_str) or str(shelves_str).strip() in ('', 'nan'): return ''
    parts = [clean_text(s.strip()) for s in str(shelves_str).split(',')]
    return ' '.join([p for p in parts if p and p not in NOISE_SHELVES])

def make_soup(row):
    title   = clean_text(row.get('title', ''))
    desc    = clean_text(row.get('description', ''))
    genres  = clean_text(row.get('genres', ''))
    author  = clean_text(row.get('author', ''))
    shelves = clean_shelves(row.get('shelves', ''))
    parts = []
    if title: parts.append(title)
    if desc: parts.append(desc)
    for _ in range(3):
        if genres: parts.append(genres)
    for _ in range(2):
        if author: parts.append(author)
    if shelves: parts.append(shelves)
    return ' '.join(parts)

if os.path.exists(SOUP_OUT):
    print(f'SKIP: {os.path.basename(SOUP_OUT)} already exists.')
else:
    print(f'Building soup for {VERSION}...')
    pf = pq.ParquetFile(BOOKS_PQ)
    writer = None
    total = 0
    for batch in tqdm(pf.iter_batches(batch_size=SOUP_CHUNK)):
        chunk = batch.to_pandas()
        soup_df = pd.DataFrame({'book_id': chunk['book_id'], 'soup': chunk.apply(make_soup, axis=1)})
        tbl = pa.Table.from_pandas(soup_df, preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(SOUP_OUT, tbl.schema)
        writer.write_table(tbl)
        total += len(chunk)
        del chunk, soup_df, tbl
        gc.collect()
    if writer: writer.close()
    print(f'Soup created for {total:,} books.')
print_ram('after soup')

Building soup for v5...


1it [00:01,  1.96s/it]

Soup created for 7,227 books.
  [RAM [after soup]] 0.30 GB


## 2. TF-IDF Fit (Sample-Based)

In [4]:
if os.path.exists(TFIDF_VEC):
    print(f'SKIP: {os.path.basename(TFIDF_VEC)} already exists.')
    with open(TFIDF_VEC, 'rb') as f:
        tfidf = pickle.load(f)
else:
    print(f'Sampling {SAMPLE_SIZE:,} texts for TF-IDF fit...')
    n_total = pq.read_metadata(SOUP_OUT).num_rows
    rng = np.random.default_rng(42)
    sample_idx = set(rng.choice(n_total, size=min(SAMPLE_SIZE, n_total), replace=False))
    sample_texts = []
    offset = 0
    pf = pq.ParquetFile(SOUP_OUT)
    for batch in tqdm(pf.iter_batches(batch_size=SOUP_CHUNK, columns=['soup'])):
        chunk = batch.to_pandas()
        for i, text in enumerate(chunk['soup']):
            if (offset + i) in sample_idx:
                sample_texts.append(text if pd.notna(text) else '')
        offset += len(chunk)
        del chunk
        gc.collect()
    print('Fitting TF-IDF vectorizer...')
    tfidf = TfidfVectorizer(stop_words='english', max_features=MAX_FEATURES, ngram_range=(1, 2), dtype=np.float32, sublinear_tf=True, min_df=2, max_df=0.95)
    tfidf.fit(sample_texts)
    del sample_texts
    gc.collect()
    with open(TFIDF_VEC, 'wb') as f:
        pickle.dump(tfidf, f, protocol=4)
    print(f'  Saved: {os.path.basename(TFIDF_VEC)}')
print_ram('after fit')

Sampling 300,000 texts for TF-IDF fit...


1it [00:00,  4.81it/s]


Fitting TF-IDF vectorizer...
  Saved: cb_tfidf_vectorizer_v5.pkl
  [RAM [after fit]] 0.38 GB


## 3. TF-IDF Transform (Chunked)

In [5]:
if os.path.exists(TFIDF_MAT):
    print(f'SKIP: {os.path.basename(TFIDF_MAT)} already exists.')
    tfidf_matrix = sp.load_npz(TFIDF_MAT)
else:
    print('Transforming in chunks...')
    sparse_chunks = []
    n_rows = 0
    pf = pq.ParquetFile(SOUP_OUT)
    for batch in tqdm(pf.iter_batches(batch_size=TFIDF_CHUNK, columns=['soup'])):
        chunk = batch.to_pandas()
        texts = chunk['soup'].fillna('').tolist()
        sub = tfidf.transform(texts)
        sparse_chunks.append(sub)
        n_rows += sub.shape[0]
        del chunk, texts
        gc.collect()
    tfidf_matrix = sp.vstack(sparse_chunks, format='csr')
    del sparse_chunks
    gc.collect()
    sp.save_npz(TFIDF_MAT, tfidf_matrix)
    print(f'  Saved: {os.path.basename(TFIDF_MAT)}')
print_ram('after transform')

Transforming in chunks...


1it [00:01,  1.77s/it]


  Saved: cb_tfidf_item_vector_v5.npz
  [RAM [after transform]] 0.42 GB


## 4. Summary

In [6]:
print('=' * 55)
print(f'  ITEM VECTOR {VERSION} SUMMARY')
print('=' * 55)
print(f'  Books in soup:     {pq.read_metadata(SOUP_OUT).num_rows:,}')
print(f'  TF-IDF shape:      {tfidf_matrix.shape}')
print(f'  TF-IDF nnz:        {tfidf_matrix.nnz:,}')
print(f'  Vocabulary size:   {len(tfidf.vocabulary_):,}')
print('=' * 55)
print_ram('final')

  ITEM VECTOR v5 SUMMARY
  Books in soup:     7,227
  TF-IDF shape:      (7227, 20000)
  TF-IDF nnz:        694,770
  Vocabulary size:   20,000
  [RAM [final]] 0.42 GB
